In [3]:
import pandas as pd
import numpy as np

data = {
    "Timestamp": pd.date_range(start="2025-01-01", periods=50000, freq="min"),
    "UserID": np.random.choice(["U1","U2","U3","U4","U5","U6","U7","U8","U9"], 50000),
    "Action": np.random.choice(["SUCCESS","ERROR"], 50000, p=[0.7,0.3]),
    "ResponseTime": np.random.randint(100, 1000, 50000)
}

df = pd.DataFrame(data)
df.to_csv("server_logs.csv", index=False)

chunk_size = 10000

user_sum = {}
user_count = {}

for chunk in pd.read_csv("server_logs.csv",
                         usecols=["Timestamp","UserID","Action","ResponseTime"],
                         chunksize=chunk_size):
    
    error_data = chunk[chunk["Action"] == "ERROR"]
    
    grouped = error_data.groupby("UserID")["ResponseTime"].agg(["sum","count"])
    
    for user in grouped.index:
        if user in user_sum:
            user_sum[user] += grouped.loc[user, "sum"]
            user_count[user] += grouped.loc[user, "count"]
        else:
            user_sum[user] = grouped.loc[user, "sum"]
            user_count[user] = grouped.loc[user, "count"]

avg_response = {user: user_sum[user]/user_count[user] for user in user_sum}

top_users = sorted(avg_response.items(), key=lambda x: x[1], reverse=True)[:7]

print("Top 7 Users:")
for user, avg in top_users:
    print(user, ":", avg)

Top 7 Users:
U3 : 553.8326309452137
U7 : 552.1841948900773
U2 : 551.9330943847073
U5 : 551.2596618357488
U1 : 548.2492736780941
U9 : 547.8231089934485
U4 : 547.1898034398034
